# Hardware calculator

A signed 32-bit integer calculator (`+ - * /`) running in the FPGA fabric, controlled over AXI-Lite from the ARM cores.

In [ ]:
from calc import CalcOverlay

ol = CalcOverlay("calc.bit")
calc = ol.calc

In [ ]:
print("7 + 5   =", calc.add(7, 5))
print("7 - 12  =", calc.sub(7, 12))
print("-6 * 7  =", calc.mul(-6, 7))
r = calc.div(-7, 2)
print("-7 / 2  =", r.value, "remainder", r.remainder)

## Edge cases

Multiply returns the full 64-bit product; the overflow flag says it does not fit in 32 bits. Division by zero is flagged by the hardware.

In [ ]:
print(calc.compute(2**31 - 1, "*", 2**31 - 1))
print(calc.compute(2**31 - 1, "+", 1))
print(calc.compute(42, "/", 0))
print(calc.compute(-2**31, "/", -1))

## Interactive

Type an expression with spaces around the operator, e.g. `123 * -45`.

In [ ]:
expr = "123 * -45"
print(expr, "=", calc.eval(expr))

## Check against Python

In [ ]:
import random

errors = 0
for _ in range(2000):
    a = random.randint(-2**31, 2**31 - 1)
    b = random.randint(-2**31, 2**31 - 1) or 1
    q = abs(a) // abs(b) * (1 if (a < 0) == (b < 0) else -1)
    expected = {"+": a + b, "-": a - b, "*": a * b, "/": q}
    for op, want in expected.items():
        got = calc.compute(a, op, b).value
        if op in "+-":
            want = (want + 2**31) % 2**32 - 2**31
        if got != want:
            errors += 1
print("errors:", errors, "of", 4 * 2000)